<a href="https://colab.research.google.com/github/vidhanj2107-bot/Diabetes-Prediction-Model-mini-project-/blob/main/Diabetes_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install lightgbm boruta imbalanced-learn

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import RandomOverSampler
from boruta import BorutaPy
import lightgbm as lgb

In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
           "Insulin", "BMI", "DiabetesPedigree", "Age", "Outcome"]

df = pd.read_csv(url, names=columns)

print("Original shape:", df.shape)
df.head()

Original shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigree,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

df[zero_cols] = df[zero_cols].replace(0, np.nan)

In [ ]:
imputer = SimpleImputer(strategy="mean")
df[zero_cols] = imputer.fit_transform(df[zero_cols])

In [ ]:
multiplier = 1.3 # stricter than 1.5

Q1 = X_temp.quantile(0.25)
Q3 = X_temp.quantile(0.75)
IQR = Q3 - Q1

mask = ~((X_temp < (Q1 - multiplier * IQR)) |
         (X_temp > (Q3 + multiplier * IQR))).any(axis=1)

X_clean = X_temp[mask]
y_clean = y_temp[mask]

print("After Strict IQR:", X_clean.shape)

After Strict IQR: (446, 8)


In [72]:
X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]

In [ ]:
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X, y)

print("Class distribution after oversampling:")
print(pd.Series(y_res).value_counts())

Class distribution after oversampling:
Outcome
1    336
0    336
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res,
    test_size=0.3,
    stratify=y_res,
    random_state=42
)

In [ ]:
rf = RandomForestClassifier(
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

boruta = BorutaPy(
    rf,
    n_estimators='auto',
    max_iter=100,
    random_state=42
)

boruta.fit(X_train.values, y_train.values)

selected_features = X_train.columns[boruta.support_].tolist()
print("Selected Features:", selected_features)

Selected Features: ['Glucose', 'BMI', 'DiabetesPedigree', 'Age']


In [ ]:
rf = RandomForestClassifier(
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

boruta = BorutaPy(
    rf,
    n_estimators='auto',
    max_iter=100,
    random_state=42
)

boruta.fit(X_train.values, y_train.values)

selected_features = X_train.columns[boruta.support_].tolist()
print("Selected Features:", selected_features)

Selected Features: ['Glucose', 'BMI', 'DiabetesPedigree', 'Age']


In [ ]:
lgb_model = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=800,
    learning_rate=0.025,
    num_leaves=35,
    min_child_samples=10,
    subsample=0.9,
    colsample_bytree=0.9
)

lgb_model.fit(X_train[selected_features], y_train)

y_pred = lgb_model.predict(X_test[selected_features])

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

[LightGBM] [Info] Number of positive: 235, number of negative: 235
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000065 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 382
[LightGBM] [Info] Number of data points in the train set: 470, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 